# Framing a prediction problem, splitting and a baseline

Core stage, lesson C19. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** name the target, the unit and the decision of a prediction problem, list what is known at the moment of prediction, split the data with a seed before looking at it, fit a baseline and a first model on the training part only, and compare them on the part that was held out.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def read(name, **kwargs):
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = read("users.csv", parse_dates=["signup_date"])
events = read("events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert (len(users), len(events)) == (21841, 177379)

# The platform of a person is the platform of their signup_complete event; its time is the signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])
users["signup_ts"] = users["user_id"].map(signup["event_ts"])
print(f"Data loaded: {len(users):,} people, {len(events):,} events.")

## Unit 1: Name the target, the unit and the decision

**Predict first.** The table of this lesson has one row for each person who signed up from January 1 to June 16 and has a signup event. The target is 1 when the person buys Plus within 14 days of signing up, else 0. How many people does the table keep, and what share of them have a 1?

In [ ]:
last_day = pd.Timestamp("2026-06-16")                 # signed up by June 16: a full 14 days before the export of July 1
first_buy = events[events["event_name"] == "purchase"].groupby("user_id")["event_ts"].min()
people = users[(users["signup_date"] <= last_day) & users["signup_ts"].notna()].copy()
people["days_to_buy"] = (people["user_id"].map(first_buy) - people["signup_ts"]).dt.total_seconds() / 86400    # NaN: never bought
people["bought14"] = (people["days_to_buy"] <= 14).astype(int)                                                    # NaN compares False: 0
print((users["signup_date"] <= last_day).sum(), len(people), bool(people["user_id"].is_unique))
print(int(people["bought14"].sum()), round(float(people["bought14"].mean()), 4))

## Unit 2: What is known at the moment

**Predict first.** The prediction moment is the instant the account exists. Apart from the signup itself, how many events does the app hold about a new person at that instant?

In [ ]:
moment = events.merge(people[["user_id", "signup_ts"]], on="user_id", validate="many_to_one")
before = moment[(moment["event_ts"] <= moment["signup_ts"]) & (moment["event_name"] != "signup_complete")]
print(len(before), len(moment), int((moment["event_name"] == "signup_complete").sum()) == len(people))
print(people[["channel", "platform", "region"]].isna().sum().to_dict())
print(round(float(moment.groupby("user_id").size().mean()), 2))

## Unit 3: Split before you look

**Predict first.** Split the people 75 to 25 with `random_state=19`, keeping the share of buyers equal in both parts. How many people go to each part, and what share of each part are buyers? Then split again with the same seed: do you get the same people? And with seed 20: how many of the first training people are in the new training part?

In [ ]:
X = people[["channel", "platform", "region"]]          # columns known at the moment
y = people["bought14"]                                 # the target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=19, stratify=y)
print(len(X_train), len(X_test))
print(round(float(y_train.mean()), 4), round(float(y_test.mean()), 4))
again = train_test_split(X, y, test_size=0.25, random_state=19, stratify=y)[0]
other = train_test_split(X, y, test_size=0.25, random_state=20, stratify=y)[0]
print(bool(again.index.equals(X_train.index)), len(X_train.index.intersection(other.index)))

## Unit 4: A baseline and a first model, fitted on the training part only

**Predict first.** The baseline gives every person the share of buyers in the training part. What chance does it give a test person? And the first model, which learns from channel, platform and region: what are the lowest and the highest chance it gives anyone in the test part: both near the baseline, or one far above it?

In [ ]:
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)      # ignores the columns: the training share of buyers for everyone
print(np.round(baseline.predict_proba(X_test.head(3))[:, 1], 4))
model = make_pipeline(
    ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel", "platform", "region"])]),   # one 0/1 column per value
    LogisticRegression(max_iter=1000),
)
model.fit(X_train, y_train)                                             # only the training part
chance = model.predict_proba(X_test)[:, 1]                              # the model's chance of a 1, for each test person
print(round(float(chance.min()), 3), round(float(chance.max()), 3))
probe = pd.DataFrame({"channel": ["referral", "paid_social"], "platform": ["web", "android"], "region": ["north_america", "north_america"]})
print(np.round(model.predict_proba(probe)[:, 1], 3))
rate = (X_train.assign(y=y_train).groupby(["channel", "platform"])["y"].mean() * 100).round(1)
print(rate.loc[[("referral", "web"), ("paid_social", "android")]].to_dict())

## Unit 5: Compare on the held-out part and write the verdict

**Predict first.** The Brier score is the average of (chance minus outcome) squared over the test people, where the outcome is 0 or 1: smaller is better. Will the first model beat the baseline on the test part by a lot, a little or not at all? And on the training part, will its score be much better than on the test part?

In [ ]:
brier_base = brier_score_loss(y_test, baseline.predict_proba(X_test)[:, 1])
brier_model = brier_score_loss(y_test, chance)
print(round(brier_base, 4), round(brier_model, 4), round(1 - brier_model / brier_base, 3))
print(round(brier_score_loss(y_train, model.predict_proba(X_train)[:, 1]), 4))          # the same score on the rows the model learned from
gaps = []
for seed in range(100, 120):                                            # twenty other splits of the same people
    a, b, c, d = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    base_s = DummyClassifier(strategy="prior").fit(a, c)
    model_s = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel", "platform", "region"])]), LogisticRegression(max_iter=1000)).fit(a, c)
    gaps.append(brier_score_loss(d, base_s.predict_proba(b)[:, 1]) - brier_score_loss(d, model_s.predict_proba(b)[:, 1]))
print(round(min(gaps), 4), round(max(gaps), 4), sum(g > 0 for g in gaps))

## Finish it

### Exercise 1: Finish it

**Task:** **How good is the first model on another split?** Split the people 80 to 20 with the seed 5 (given), fit the recipe on the training part only, and score the chances it gives the test part with the Brier score. Put the score, rounded to 4 decimals, in `answer_1`.

**Expected:** `answer_1` is a number between 0 and 1.

In [ ]:
X5_train, X5_test, y5_train, y5_test = train_test_split(X, y, test_size=0.2, random_state=5, stratify=y)    # given: an 80 to 20 split with seed 5
recipe = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel", "platform", "region"])]), LogisticRegression(max_iter=1000))    # given: the recipe of unit 4
# fit the recipe on the training part only, then score the chances it gives the test part
answer_1 = None  # replace None with the score, rounded to 4 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", 0.1013, traps={0.1012: "That is the score on the training part. Score the test part.", 0.1058: "That is the score of the baseline. Score the recipe.", 0.1202: "That scores hard 0 and 1 answers. Give the Brier score the chances from predict_proba."}, tol=5e-5)

<details>
<summary>Hint</summary>

Fit with `recipe.fit(X5_train, y5_train)`. Then `recipe.predict_proba(X5_test)[:, 1]` gives the chance of a 1 for each test person: the same two calls as in unit 4.

</details>

<details>
<summary>Half-solution</summary>

```python
recipe.fit(..., ...)
answer_1 = round(float(brier_score_loss(..., recipe.predict_proba(...)[:, 1])), 4)
```

</details>

<details>
<summary>Solution</summary>

```python
recipe.fit(X5_train, y5_train)
answer_1 = round(float(brier_score_loss(y5_test, recipe.predict_proba(X5_test)[:, 1])), 4)
```

On this split the first model scores 0.1013 on 4,035 test people. The baseline of this split scores 0.1058, so the gap is about the size of the gaps of unit 5. The check cell reports the score on the training part (0.1012) or the baseline's, if you give one of those.

</details>

## On your own

### Exercise 2: On your own

**Task:** **How much of the gain comes from the channel alone?** Before you start, say in one sentence what the result looks like: two numbers, both between 0.09 and 0.12, the second one larger than the first. Use the split of Exercise 1. Fit the recipe of unit 4 on the `channel` column alone, on the training part, and put the list [its Brier score on the test part, the Brier score of the baseline on the test part] in `answer_2`, each rounded to 4 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.1014, 0.1058], traps={(0.1013, 0.1058): "That uses all three columns. Give the recipe the channel column alone.", (0.1058, 0.1014): "The model's score comes first, the baseline's second."}, tol=5e-5)

<details>
<summary>Hint</summary>

Copy the recipe of unit 4, with `["channel"]` in place of the three column names, and fit it on `X5_train[["channel"]]`. The baseline is the `DummyClassifier` of unit 4.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Two numbers a little apart: the model's score about 0.101, the baseline's about 0.106."*

```python
chan_model = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel"])]), LogisticRegression(max_iter=1000))
chan_model.fit(..., y5_train)
base2 = DummyClassifier(strategy="prior").fit(..., y5_train)
```

</details>

<details>
<summary>Solution</summary>

```python
chan_model = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel"])]), LogisticRegression(max_iter=1000))
chan_model.fit(X5_train[["channel"]], y5_train)
base2 = DummyClassifier(strategy="prior").fit(X5_train[["channel"]], y5_train)
score_chan = brier_score_loss(y5_test, chan_model.predict_proba(X5_test[["channel"]])[:, 1])
score_base = brier_score_loss(y5_test, base2.predict_proba(X5_test[["channel"]])[:, 1])
answer_2 = [round(float(score_chan), 4), round(float(score_base), 4)]
```

The channel alone scores 0.1014 against 0.1058 for the baseline. The model with all three columns (Exercise 1) scores 0.1013, so platform and region add 0.0001 on this split: nearly all of the gain is the channel. Fewer columns would do the same job, and a column is also something that must be known at the moment of prediction.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague compares the first model with a baseline on the split of Exercise 1 and reports that the model is 16 percent better. The baseline is not fair. Fix the baseline so that the comparison is fair, and set `answer_3` to the gain `1 - model score / baseline score`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
model3 = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel", "platform", "region"])]), LogisticRegression(max_iter=1000)).fit(X5_train, y5_train)     # given: the first model, fitted on the training part
base_bad = DummyClassifier(strategy="most_frequent").fit(X5_train, y5_train)       # colleague: "the baseline predicts the most frequent outcome"
b_base = brier_score_loss(y5_test, base_bad.predict_proba(X5_test)[:, 1])
b_model = brier_score_loss(y5_test, model3.predict_proba(X5_test)[:, 1])
answer_3 = round(float(1 - b_model / b_base), 3)

In [ ]:
_ = check("Exercise 3", "answer_3", 0.042, traps={0.158: "That gain is against a baseline that gives every person a chance of 0. A fair baseline gives every person the training share of buyers.", 0.0: "Check the sign and the order: the model's score is divided by the baseline's."}, tol=5e-4)

<details>
<summary>Hint</summary>

Print `base_bad.predict_proba(X5_test)[:3]`. What chance of a 1 does this baseline give a person? What chance does the baseline of unit 4 give?

</details>

<details>
<summary>Half-solution</summary>

```python
base_ok = DummyClassifier(strategy=...).fit(X5_train, y5_train)
b_base = brier_score_loss(y5_test, base_ok.predict_proba(X5_test)[:, 1])
```

</details>

<details>
<summary>Solution</summary>

```python
base_ok = DummyClassifier(strategy="prior").fit(X5_train, y5_train)
b_base = brier_score_loss(y5_test, base_ok.predict_proba(X5_test)[:, 1])
b_model = brier_score_loss(y5_test, model3.predict_proba(X5_test)[:, 1])
answer_3 = round(float(1 - b_model / b_base), 3)
```

The gain is 0.042, not 0.158. The `most_frequent` baseline answers "never buys" with a chance of 0 for everyone, so it pays the full squared miss of 1 on each of the 12 percent who buy: a score of 0.1202, which is simply the share of buyers. The honest baseline gives everyone the chance 0.1203 (the training share) and scores 0.1058. A baseline weaker than the simplest honest answer makes any model look good.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")